# BPM2D KV260 Kerr Interleaved SI

Interactive inspection of the physically consistent `64x64` PYNQ overlay. The official timing source is the headless benchmark; this notebook invokes the same code and visualizes its saved results.

In [ ]:
from pathlib import Path
import csv
import json
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
metadata = json.loads((ROOT / "data" / "reference_meta.json").read_text())
metadata

In [ ]:
phi_in = np.load(ROOT / "data" / "phi_in_64.npy")
masks = [np.load(ROOT / "data" / f"phase_factor_{i}.npy") for i in range(1, 4)]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), constrained_layout=True)
images = [np.abs(phi_in), *[np.angle(mask) for mask in masks]]
titles = ["Input magnitude", "Mask 1 phase", "Mask 2 phase", "Mask 3 phase"]
for axis, image, title in zip(axes, images, titles):
    rendered = axis.imshow(image, origin="lower", cmap="viridis")
    axis.set_title(title)
    axis.set_xlabel("x sample")
    axis.set_ylabel("y sample")
    fig.colorbar(rendered, ax=axis, shrink=0.78)
plt.show()

## Smoke test

This programs the overlay, validates all six reference states, performs one warm-up PSF, and measures three PSFs.

In [ ]:
from benchmark_bpm2d_kerr_interleaved_si import run_benchmark

smoke_summary = run_benchmark(ROOT, warmups=1, repetitions=3)
smoke_summary

## Official run

For the reported measurement, execute the headless script in a terminal. The following command uses the same required 5 warm-up and 100 measured PSFs:

```bash
python3 benchmark_bpm2d_kerr_interleaved_si.py --warmups 5 --repetitions 100
```

In [ ]:
result_dir = Path(smoke_summary["result_directory"])
validation = json.loads((result_dir / "validation.json").read_text())
validation

In [ ]:
with (result_dir / "repetitions.csv").open(newline="") as stream:
    rows = list(csv.DictReader(stream))

kernel = np.array([float(row["kernel_total_s"]) for row in rows])
end_to_end = np.array([float(row["end_to_end_s"]) for row in rows])
indices = np.arange(len(rows))

fig, axis = plt.subplots(figsize=(8, 3.8), constrained_layout=True)
axis.plot(indices, kernel, marker="o", label="Three PL calls")
axis.plot(indices, end_to_end, marker="s", label="End-to-end PSF")
axis.set_xlabel("Measured repetition")
axis.set_ylabel("Time (s)")
axis.grid(True, alpha=0.3)
axis.legend()
plt.show()

In [ ]:
final_field = np.load(result_dir / "final_output.npy")
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), constrained_layout=True)
for axis, image, title in zip(
    axes,
    [np.abs(final_field), np.angle(final_field)],
    ["Final magnitude", "Final phase"],
):
    rendered = axis.imshow(image, origin="lower", cmap="viridis")
    axis.set_title(title)
    axis.set_xlabel("x sample")
    axis.set_ylabel("y sample")
    fig.colorbar(rendered, ax=axis, shrink=0.8)
plt.show()